# Preprocessing and Consolidation of Experimental Outputs

This notebook integrates multiple sources of model outputs, including image rankings, labels, and captions, in order to generate structured files for downstream evaluation and analysis.

The pipeline is designed to be **model-agnostic**, meaning that different ranking models (e.g., ResNet, DINOv2, ViT) can be used interchangeably by simply switching the input files.

## Input Files

The notebook expects the following inputs:

- `corel5k_lists.txt`: list of all images in the dataset
- Ranking file (model-dependent):
  - ResNet: `CNN-ResNet.7z`
  - DINOv2: `corel5k_dinov2_vitg14.txt`
  - ViT-B/16: `rks_VIT-B16_original_corel5k.txt`

Only **one ranking source is used per execution**, depending on the model being evaluated.

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess

DATA_DIR = Path("../data")

# -----------------------------
# Select ranking model
# -----------------------------

RANK_FILE = DATA_DIR / "CNN-ResNet.txt"
# RANK_FILE = DATA_DIR / "corel5k_dinov2_vitg14.txt"
# RANK_FILE = DATA_DIR / "rks_VIT-B16_original_corel5k.txt"

LIST_FILE = DATA_DIR / "corel5k_lists.txt"

Mounted at /content/drive


In [ ]:
# If using compressed file (e.g., ResNet .7z)
# Run only when necessary

subprocess.run(["7z", "x", str(DATA_DIR / "CNN-ResNet.7z"), f"-o{DATA_DIR}"], check=True)


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (406F0),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/                   1 file, 43945075 bytes (42 MiB)

Extracting archive: /content/CNN-DPNet.7z
--
Path = /content/CNN-DPNet.7z
Type = 7z
Physical Size = 43945075
Headers Size = 130
Method = LZMA2:24
Solid = -
Blocks = 1

  0%      4% - CNN-DPNet.txt                      9% - CNN-DPNet.txt                     14% - CNN-DPNet.txt                     18% - CNN-DPNet.txt                     23% - CNN-DPNet.txt                     28% - CNN-DPNet.txt                    

In [ ]:
with open(RANK_FILE, "r") as f:
    rank_lines = [line.strip().split() for line in f]

with open(LIST_FILE, "r") as f:
    image_list = [line.strip() for line in f]

idx_to_image = {i: name for i, name in enumerate(image_list)}

results = []

for i, line in enumerate(rank_lines):
    image = idx_to_image[i]

    ranked_images = [
        idx_to_image[int(idx)]
        for idx in line
    ]

    results.append([
        image,
        " ".join(ranked_images)
    ])

df_rank = pd.DataFrame(results, columns=["image", "ranked_images"])

Arquivo 'rank_swintf.csv' criado!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,imagem,imagens_rankeadas
0,1.jpg,1.jpg 9.jpg 5.jpg 56.jpg 3.jpg 55.jpg 84.jpg 3...
1,10.jpg,10.jpg 22.jpg 33.jpg 84.jpg 8.jpg 23.jpg 68.jp...
2,100.jpg,100.jpg 78.jpg 76.jpg 79.jpg 81.jpg 11.jpg 97....
3,1000.jpg,1000.jpg 989.jpg 928.jpg 960.jpg 966.jpg 901.j...
4,1001.jpg,1001.jpg 1065.jpg 1097.jpg 1081.jpg 1096.jpg 1...
...,...,...
4995,995.jpg,995.jpg 987.jpg 994.jpg 960.jpg 975.jpg 908.jp...
4996,996.jpg,996.jpg 941.jpg 914.jpg 960.jpg 928.jpg 966.jp...
4997,997.jpg,997.jpg 992.jpg 1000.jpg 989.jpg 2150.jpg 994....
4998,998.jpg,998.jpg 999.jpg 3554.jpg 937.jpg 973.jpg 2159....


In [ ]:
OUTPUT_FILE = DATA_DIR / "rank_resnet.csv"

df_rank.to_csv(OUTPUT_FILE, index=False)

print(f"File saved at: {OUTPUT_FILE}")

Linhas: 5000
Imagens únicas: 5000


In [ ]:
df_check = pd.read_csv(OUTPUT_FILE)

# 5000
print("Rows:", len(df_check))
print("Unique images:", df_check["image"].nunique())
df_check.head()

## Ranked Caption Generation

This notebook merges image ranking files with automatically generated captions (BLIP outputs) to produce a structured dataset where captions are organized according to model-based ranking.

This allows analyzing caption quality in a **rank-aware manner**, enabling downstream evaluation of semantic consistency across retrieval orderings.

### Model Flexibility

The ranking source can be swapped depending on the visual encoder used:

- ResNet
- DINOv2
- ViT-B/16

Only the input ranking file changes; the rest of the pipeline remains unchanged.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

# -----------------------------
# Select ranking model
# -----------------------------
RANK_FILE = DATA_DIR / "rank_resnet.csv"
# RANK_FILE = DATA_DIR / "rank_dinov2.csv"
# RANK_FILE = DATA_DIR / "rank_vit.csv"

CAPTIONS_FILE = DATA_DIR / "captions.csv"

In [ ]:
captions = pd.read_csv(CAPTIONS_FILE)
ranks = pd.read_csv(RANK_FILE)

# Map: image -> caption
caption_dict = dict(zip(captions.iloc[:, 0], captions.iloc[:, 1]))

print("Number of captions:", len(caption_dict))

In [ ]:
result = {}
counter = 0

for _, row in ranks.iterrows():
    counter += 1

    if counter % 500 == 0:
        print(f"Processed {counter} rows")

    base_image = row["image"]
    ranked_images = str(row["ranked_images"]).split()

    base_caption = caption_dict.get(base_image, "")
    column_name = f"{base_image}: {base_caption}"

    ranked_captions = [
        caption_dict.get(img, "")
        for img in ranked_images
    ]

    result[column_name] = ranked_captions

# result: 5000
print("Columns created:", len(result))

In [ ]:

OUTPUT_FILE = DATA_DIR / "captions_rank_resnet.csv"

df_final = pd.DataFrame.from_dict(result, orient="index").T

df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved file to: {OUTPUT_FILE}")

5000
Linha 500 processada
Linha 1000 processada
Linha 1500 processada
Linha 2000 processada
Linha 2500 processada
Linha 3000 processada
Linha 3500 processada
Linha 4000 processada
Linha 4500 processada
Linha 5000 processada
Colunas criadas: 5000
Arquivo 'legendas_rank_swintf.csv' criado!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,"1.jpg: ""a stained glass window with a circular design""","10.jpg: ""a stained glass window with a painting of a man holding a child""","100.jpg: ""a church lit up at night""","1000.jpg: ""a bunch of green and red onions""","1001.jpg: ""a woman standing in a field with her hands on her hips""","1002.jpg: ""a woman sitting on the ground with her legs crossed""","1003.jpg: ""a woman sitting on a bench""","1004.jpg: ""a woman in a hat and blue jeans""","1005.jpg: ""a woman in a black coat and boots""","1006.jpg: ""a woman walking down a street with a cell""",...,"990.jpg: ""oki oki oki oki oki oki oki oki oki oki""","991.jpg: ""a red apple""","992.jpg: ""three green pears""","993.jpg: ""a piece of food""","994.jpg: ""a bunch of onions on a black background""","995.jpg: ""a plant with green leaves on it""","996.jpg: ""a pile of green leaves""","997.jpg: ""a bowl filled with apples and a bunch of leaves""","998.jpg: ""a variety of beans and beans""","999.jpg: ""a pile of green peas"""
0,"""a stained glass window with a circular design""","""a stained glass window with a painting of a m...","""a church lit up at night""","""a bunch of green and red onions""","""a woman standing in a field with her hands on...","""a woman sitting on the ground with her legs c...","""a woman sitting on a bench""","""a woman in a hat and blue jeans""","""a woman in a black coat and boots""","""a woman walking down a street with a cell""",...,"""oki oki oki oki oki oki oki oki oki oki""","""a red apple""","""three green pears""","""a piece of food""","""a bunch of onions on a black background""","""a plant with green leaves on it""","""a pile of green leaves""","""a bowl filled with apples and a bunch of leaves""","""a variety of beans and beans""","""a pile of green peas"""
1,"""a stained glass window with a circular design""","""a stained glass window with a man holding a b...","""a stained glass window""","""a red flower with a black background""","""a woman in a plaid skirt standing in a field""","""a woman sitting on the floor with her legs cr...","""a woman with long hair and a red jacket""","""a woman in a hat and blue shirt""","""a woman walking down a street with a cell""","""a woman in a black coat and boots""",...,"""two cus with green leaves and white spots""","""a bunch of onions on a black background""","""a fruit with a half cut in half""","""a bunch of onions on a black background""","""a red apple""","""a white flower with green leaves on it""","""a close up of a green leaf""","""three green pears""","""a pile of green peas""","""a pile of white beans"""
2,"""a stained glass window with a cross in the ce...","""a stained glass window with a man and woman""","""two windows with stained glass in them""","""a large red and green vegetable on a green leaf""","""a woman sitting in a field with her hand on h...","""a woman sitting on the ground with her legs c...","""a woman sitting on a bench in a park""","""a woman wearing a hat and a vest""","""a woman sitting on a rock in the grass""","""a woman sitting on a rock in the grass""",...,"""a melon melon melon melon melon melon melon m...","""a red onion on a black background""","""three peppers are shown in a row""","""two white mushrooms in the ground""","""two cabbages with the same leaves""","""a bunch of onions on a black background""","""a green leaf with a black background""","""a bunch of green and red onions""","""a pile of white beans""","""a variety of beans and beans"""
3,"""a stained glass window with a statue in it""","""a stained glass window with a nat scene""","""a stained glass window""","""a bunch of green leaves""","""a woman standing in front of a tree""","""a woman sitting on a chair""","""a woman with a smile""","""a woman in a blue dress and brown hat""","""a woman sitting on a bench in a park""","""a woman sitting on a bench in a park""",...,"""a green cunt with a black background""","""two apples with green leaves on them""","""a bowl filled with apples and a bunch 